# Unidad 3: RDDs — Resilient Distributed Datasets

**Tecnicatura en Datos – Procesamiento con Apache Spark**  
Unidad 3 de 10 — Duración estimada: 2:30 hs

> **Nota Databricks:** `spark` y `sc` ya están disponibles. No es necesario crearlos.

In [ ]:
# Obtener el SparkContext desde la SparkSession
sc = spark.sparkContext
print(f"SparkContext activo | Versión: {sc.version} | Master: {sc.master}")

---
## 1. ¿Qué es un RDD?

**RDD = Resilient Distributed Dataset**

| Característica | Significado |
|---|---|
| **Resilient** | Tolerante a fallos (se recupera vía lineage) |
| **Distributed** | Los datos se distribuyen en múltiples nodos |
| **Dataset** | Colección de elementos |

Propiedades clave:
- **Inmutabilidad**: cada transformación produce un nuevo RDD
- **Particionado**: los datos se dividen en particiones procesadas en paralelo
- **Lineage**: Spark guarda el historial de cómo fue creado, no los datos

---
## 2. Creación de RDDs

### Ejemplo 1 — Simple: lista de números

In [ ]:
# Creamos un RDD desde una lista en memoria
numeros = sc.parallelize([1, 2, 3, 4, 5])

print("Número de particiones:", numeros.getNumPartitions())
print("Elementos:", numeros.collect())

**Resultado esperado:**
```
Número de particiones: 4          # varía según los cores disponibles
Elementos: [1, 2, 3, 4, 5]
```

> `parallelize` distribuye la lista entre los workers. Ideal para testing, no para grandes volúmenes.

### Ejemplo 2 — Medio: RDD de tuplas con `reduceByKey`

In [ ]:
# RDD de ventas (producto, monto)
ventas = sc.parallelize([
    ("manzana", 100),
    ("banana",  200),
    ("manzana", 150),
    ("banana",   50),
    ("naranja", 300),
])

# Suma de montos por producto
total_por_producto = ventas.reduceByKey(lambda a, b: a + b)

print("Total por producto:", total_por_producto.collect())

**Resultado esperado:**
```
Total por producto: [('manzana', 250), ('banana', 250), ('naranja', 300)]
```

> `reduceByKey` agrupa por la primera posición de la tupla y aplica la función acumuladora.

### Ejemplo 3 — Avanzado: control de particiones con `glom()`

In [ ]:
# 20 elementos distribuidos en 4 particiones
datos = sc.parallelize(range(1, 21), numSlices=4)

# glom() convierte cada partición en una lista para poder inspeccionarla
particiones = datos.glom().collect()
for i, part in enumerate(particiones):
    print(f"Partición {i}: {part}")

print("\nTotal elementos:", datos.count())

**Resultado esperado:**
```
Partición 0: [1, 2, 3, 4, 5]
Partición 1: [6, 7, 8, 9, 10]
Partición 2: [11, 12, 13, 14, 15]
Partición 3: [16, 17, 18, 19, 20]
Total elementos: 20
```

> `glom()` es muy útil para diagnosticar **skew** (desbalance de datos entre particiones).

---
## 3. Lazy Evaluation

In [ ]:
import time

rdd_base = sc.parallelize(range(1, 1_000_001))

t0 = time.time()
# Estas transformaciones NO ejecutan nada — solo definen el plan
rdd_filtrado = rdd_base.filter(lambda x: x % 2 == 0)
rdd_mapeado  = rdd_filtrado.map(lambda x: x * 2)
t1 = time.time()
print(f"Definir transformaciones: {t1-t0:.4f}s  ← casi 0")

t2 = time.time()
# count() es una ACCIÓN — aquí se ejecuta todo el plan
resultado = rdd_mapeado.count()
t3 = time.time()
print(f"Ejecutar con count():     {t3-t2:.4f}s  ← procesamiento real")
print(f"Resultado: {resultado}")

---
## 4. Transformaciones

### Ejemplo 1 — Simple: `filter` y `map`

In [ ]:
numeros = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8])

pares      = numeros.filter(lambda x: x % 2 == 0)  # Transformación 1
duplicados = pares.map(lambda x: x * 2)             # Transformación 2

# collect() es la ACCIÓN que dispara el plan
print("Resultado:", duplicados.collect())

**Resultado esperado:**
```
Resultado: [4, 8, 12, 16]
```

### Ejemplo 2 — Medio: `map` vs `flatMap`

In [ ]:
frases = sc.parallelize([
    "Spark es rápido",
    "RDD es la base",
    "flatMap aplana listas",
])

# map: 1 elemento → 1 elemento (una lista por frase)
con_map = frases.map(lambda f: f.split(" "))
print("map    →", con_map.collect())

# flatMap: 1 elemento → N elementos (palabras sueltas)
con_flatmap = frases.flatMap(lambda f: f.split(" "))
print("flatMap →", con_flatmap.collect())

**Resultado esperado:**
```
map    → [['Spark', 'es', 'rápido'], ['RDD', 'es', 'la', 'base'], ['flatMap', 'aplana', 'listas']]
flatMap → ['Spark', 'es', 'rápido', 'RDD', 'es', 'la', 'base', 'flatMap', 'aplana', 'listas']
```

### Ejemplo 3 — Avanzado: `union`, `distinct`, `sortBy`

In [ ]:
ventas_enero   = sc.parallelize(["manzana", "banana", "naranja", "manzana"])
ventas_febrero = sc.parallelize(["banana", "uva", "naranja", "kiwi"])

todas    = ventas_enero.union(ventas_febrero)   # concatena (con duplicados)
unicos   = todas.distinct()                     # elimina duplicados
ordenado = unicos.sortBy(lambda x: x)           # ordena

print("Productos únicos (ambos meses):", ordenado.collect())

**Resultado esperado:**
```
Productos únicos (ambos meses): ['banana', 'kiwi', 'manzana', 'naranja', 'uva']
```

---
## 5. Acciones

### Ejemplo 1 — Simple: `count()` y `take()`

In [ ]:
numeros = sc.parallelize(range(1, 101))

print("Total de elementos:", numeros.count())
print("Primeros 5:",         numeros.take(5))

**Resultado esperado:**
```
Total de elementos: 100
Primeros 5: [1, 2, 3, 4, 5]
```

### Ejemplo 2 — Medio: `reduce()`

In [ ]:
numeros = sc.parallelize([1, 2, 3, 4, 5])

suma     = numeros.reduce(lambda a, b: a + b)
producto = numeros.reduce(lambda a, b: a * b)

print("Suma:    ", suma)
print("Producto:", producto)

**Resultado esperado:**
```
Suma:     15
Producto: 120
```

> `reduce` debe usar funciones **asociativas y conmutativas** para funcionar correctamente en entornos distribuidos.

### Ejemplo 3 — Avanzado: `aggregate()` — múltiples acumuladores en una pasada

In [ ]:
numeros = sc.parallelize([10, 20, 30, 40, 50])

# Valor inicial: (suma_acumulada, conteo)
cero = (0, 0)

def combinar(acc, valor):
    """Combina el acumulador con un elemento de la partición."""
    return (acc[0] + valor, acc[1] + 1)

def fusionar(acc1, acc2):
    """Combina dos acumuladores entre particiones."""
    return (acc1[0] + acc2[0], acc1[1] + acc2[1])

suma_total, cantidad = numeros.aggregate(cero, combinar, fusionar)
promedio = suma_total / cantidad

print(f"Suma: {suma_total} | Cantidad: {cantidad} | Promedio: {promedio}")

**Resultado esperado:**
```
Suma: 150 | Cantidad: 5 | Promedio: 30.0
```

> `aggregate` recorre el RDD **una sola vez** para calcular múltiples métricas. Más eficiente que llamar `sum()` y `count()` por separado.

---
## 6. Lineage y tolerancia a fallos

In [ ]:
# Ver el lineage de un RDD
rdd_base    = sc.parallelize(range(1, 11))
rdd_filtro  = rdd_base.filter(lambda x: x % 2 == 0)
rdd_mapeado = rdd_filtro.map(lambda x: x ** 2)

# toDebugString muestra el árbol de dependencias (lineage)
print("=== Lineage del RDD ===")
print(rdd_mapeado.toDebugString().decode("utf-8"))

**Resultado esperado:**
```
(4) PythonRDD[3] at RDD at PythonRDD.scala:53 []
 |  PythonRDD[2] at RDD at PythonRDD.scala:53 []
 |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:289 []
```

> Si una partición se pierde, Spark usa el lineage para recalcularla sin necesidad de replicar los datos.

---
## 7. Caso clásico: Word Count

### Ejemplo 1 — Simple: lista en memoria

In [ ]:
texto = sc.parallelize([
    "hola mundo",
    "hola spark",
    "spark es rapido",
])

conteo = (
    texto
    .flatMap(lambda linea: linea.split(" "))
    .map(lambda palabra: (palabra, 1))
    .reduceByKey(lambda a, b: a + b)
)

print(conteo.collect())

**Resultado esperado:**
```
[('hola', 2), ('mundo', 1), ('spark', 2), ('es', 1), ('rapido', 1)]
```

### Ejemplo 2 — Medio: Word Count normalizado y ordenado

In [ ]:
# Simulación de lectura desde archivo
# En Databricks con HDFS: sc.textFile("/mnt/datalake/textos.txt")
lineas = sc.parallelize([
    "Apache Spark es un motor de procesamiento distribuido",
    "Spark usa la memoria RAM para procesar datos más rápido",
    "Apache Hadoop usa MapReduce y escribe en disco",
    "Spark es compatible con Hadoop y con S3",
])

conteo = (
    lineas
    .flatMap(lambda linea: linea.split(" "))
    .map(lambda p: p.lower().strip())
    .filter(lambda p: len(p) > 2)           # eliminar palabras muy cortas
    .map(lambda p: (p, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda par: par[1], ascending=False)
)

print("Top 10 palabras:")
for palabra, freq in conteo.take(10):
    print(f"  {palabra:<15} → {freq}")

**Resultado esperado:**
```
Top 10 palabras:
  spark           → 3
  apache          → 2
  hadoop          → 2
  con             → 2
  ...
```

### Ejemplo 3 — Avanzado: Word Count con stopwords y persistencia

In [ ]:
stopwords = {"de", "la", "el", "en", "y", "a", "los", "las", "un", "una", "es",
             "con", "por", "para", "que", "del", "más", "sus"}

corpus = sc.parallelize([
    "El procesamiento distribuido con Apache Spark es eficiente",
    "Spark usa la memoria para procesar datos en paralelo",
    "Los RDDs son la base del modelo de Spark",
    "El lineage de Spark permite tolerancia a fallos sin replicación",
    "Los DataFrames son más rápidos que los RDDs en la mayoría de casos",
] * 100)  # simular corpus más grande

palabras_limpias = (
    corpus
    .flatMap(lambda linea: linea.lower().split())
    .map(lambda p: ''.join(c for c in p if c.isalpha()))  # solo letras
    .filter(lambda p: p and p not in stopwords)
    .map(lambda p: (p, 1))
    .reduceByKey(lambda a, b: a + b)
)

# Persistir porque lo vamos a usar dos veces
palabras_limpias.persist()

top20 = palabras_limpias.sortBy(lambda par: par[1], ascending=False).take(20)
print(f"Top 20 palabras: {top20}")
print(f"Vocabulario único: {palabras_limpias.count()} palabras")

palabras_limpias.unpersist()

**Resultado esperado:**
```
Top 20 palabras: [('spark', 500), ('datos', 300), ('rdds', 200), ...]
Vocabulario único: 28 palabras
```

> `.persist()` evita que Spark recalcule el RDD desde cero en el segundo uso. `unpersist()` libera la memoria cuando ya no se necesita.

---
## 8. Práctica guiada

### Ejercicio 1 — Simple: temperaturas

In [ ]:
temperaturas = sc.parallelize([22.5, 31.0, 28.3, 35.1, 29.9, 33.4, 19.0, 30.0, 27.6, 36.2])

dias_calurosos = temperaturas.filter(lambda t: t > 30)
cantidad = dias_calurosos.count()
maxima   = temperaturas.reduce(lambda a, b: a if a > b else b)

print(f"Días con más de 30°C: {cantidad}")
print(f"Temperatura máxima:   {maxima}°C")

**Resultado esperado:**
```
Días con más de 30°C: 4
Temperatura máxima:   36.2°C
```

### Ejercicio 2 — Medio: análisis de órdenes de compra

In [ ]:
ordenes = sc.parallelize([
    "laptop,mouse,teclado",
    "monitor,auriculares",
    "laptop,webcam,mouse",
    "teclado",
])

todos_productos = ordenes.flatMap(lambda o: o.split(","))

# Productos únicos
unicos = todos_productos.distinct().sortBy(lambda x: x)
print("Productos únicos:", unicos.collect())

# Frecuencia de cada producto
frecuencia = (
    todos_productos
    .map(lambda p: (p, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)
print("Frecuencia:", frecuencia.collect())

**Resultado esperado:**
```
Productos únicos: ['auriculares', 'laptop', 'monitor', 'mouse', 'teclado', 'webcam']
Frecuencia: [('teclado', 2), ('laptop', 2), ('mouse', 2), ('monitor', 1), ('auriculares', 1), ('webcam', 1)]
```

### Ejercicio 3 — Avanzado: análisis de logs

In [ ]:
logs = sc.parallelize([
    "INFO  | 2024-01-10 08:00 | Servicio iniciado correctamente",
    "WARN  | 2024-01-10 08:05 | Memoria al 80%",
    "ERROR | 2024-01-10 08:07 | Conexión rechazada en puerto 5432",
    "INFO  | 2024-01-10 08:10 | Solicitud procesada",
    "ERROR | 2024-01-10 08:12 | Timeout al conectar con el servicio de pagos",
    "WARN  | 2024-01-10 08:15 | CPU al 90%",
    "INFO  | 2024-01-10 08:20 | Caché limpiado",
    "ERROR | 2024-01-10 08:22 | NullPointerException en módulo de reportes",
])

def parsear(linea):
    partes = linea.split(" | ")
    return (partes[0].strip(), partes[1].strip(), partes[2].strip())

logs_parseados = logs.map(parsear)
logs_parseados.persist()

# 1. Conteo por nivel
conteo_niveles = (
    logs_parseados
    .map(lambda r: (r[0], 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[0])
)
print("Eventos por nivel:", conteo_niveles.collect())

# 2. Mensajes de ERROR
errores = (
    logs_parseados
    .filter(lambda r: r[0] == "ERROR")
    .map(lambda r: f"[{r[1]}] {r[2]}")
)
print("\nMensajes de ERROR:")
for e in errores.collect():
    print(f"  - {e}")

# 3. Tasa de error
total  = logs_parseados.count()
nerror = logs_parseados.filter(lambda r: r[0] == "ERROR").count()
print(f"\nTasa de error: {nerror}/{total} = {nerror/total:.1%}")

logs_parseados.unpersist()

**Resultado esperado:**
```
Eventos por nivel: [('ERROR', 3), ('INFO', 3), ('WARN', 2)]

Mensajes de ERROR:
  - [2024-01-10 08:07] Conexión rechazada en puerto 5432
  - [2024-01-10 08:12] Timeout al conectar con el servicio de pagos
  - [2024-01-10 08:22] NullPointerException en módulo de reportes

Tasa de error: 3/8 = 37.5%
```

---
## 9. Preguntas de revisión

1. ¿Por qué los RDDs son inmutables?
2. ¿Qué ventaja tiene el lineage frente a la replicación?
3. ¿Cuál es la diferencia entre `map` y `flatMap`?
4. ¿Cuándo conviene usar `aggregate` en lugar de `sum` + `count`?
5. ¿Cuándo conviene usar RDD en lugar de DataFrame?

---
**Próxima unidad:** DataFrames y Spark SQL